# Turn Flavor & Rationale Explorer

Explores AI flavor settings and corresponding rationale text from `turn_data.csv`.
Focus: when does `use_nuke` first reach 100, and what did the AI say?

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')
from plot_utilities import load_turn_data, setup_notebook_display
setup_notebook_display()

In [2]:
df = load_turn_data('../turn_data.csv', condition_exclude=['observe-vanilla-standard'])

d:\Cache\Onedrive - Arizona\OneDrive - University of Arizona\Vox Deorum\nlp-2026-tiny-4player\analysis\exploratory\..\plot_utilities.py:401: DtypeWarning: Columns (63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


✓ Loaded turn data: 747024 rows
✓ Filters applied: condition_exclude=['observe-vanilla-standard']
✓ Unique conditions: 11
✓ Unique player types: 13
✓ Turn range: 0 - 500
✓ Unique civilizations: 43

Player Type Distribution:
  Vanilla: 257290 rows (34.4%)
  GLM-4.7-Simple: 73618 rows (9.9%)
  GLM-4.7-Briefed: 73618 rows (9.9%)
  Kimi-K2.5-Briefed: 49525 rows (6.6%)
  Kimi-K2.5-Simple: 49525 rows (6.6%)
  GPT-OSS-120B-Briefed: 49211 rows (6.6%)
  GPT-OSS-120B-Simple: 49211 rows (6.6%)
  Deepseek-3.2-Simple: 47818 rows (6.4%)
  Deepseek-3.2-Briefed: 47818 rows (6.4%)
  Minimax-M2.5-Briefed: 13503 rows (1.8%)
  Minimax-M2.5-Simple: 13503 rows (1.8%)
  Sonnet-4.5-Briefed: 11192 rows (1.5%)
  Sonnet-4.5-Simple: 11192 rows (1.5%)

Condition Distribution:
  2026-deepseek-v-kimi2-v-glm-standard: 281688 rows (37.7%)
  2026-oss-v-glm-standard: 223776 rows (30.0%)
  2026-deepseek-v-kimi2-v-minimax-standard: 56856 rows (7.6%)
  2026-oss-v-sonnet-standard: 45584 rows (6.1%)
  2026-oss-v-minimax-stan

## Flavor overview & first `use_nuke = 100` per game

In [3]:
flavor_cols = [c for c in df.columns if c.startswith('flavor_')]
print(f'Flavor columns ({len(flavor_cols)}): {flavor_cols}')

changed = df[df['is_changed'] == 1].copy()
print(f'\nTotal rows: {len(df):,}  |  Changed turns: {len(changed):,}')

nuke_esc = changed[changed['flavor_use_nuke'] == 100]

first_nuke = (
    nuke_esc
    .sort_values('turn')
    .groupby(['game_id', 'player_id'], as_index=False)
    .first()
)

print(f'Players who ever set use_nuke=100: {len(first_nuke)}')
pd.set_option('display.max_colwidth', 120)
first_nuke[['player_type', 'civilization', 'turn', 'grand_strategy', 'flavor_use_nuke', 'rationale']].head(10)

Flavor columns (34): ['flavor_offense', 'flavor_defense', 'flavor_mobilization', 'flavor_city_defense', 'flavor_military_training', 'flavor_recon', 'flavor_ranged', 'flavor_mobile', 'flavor_nuke', 'flavor_use_nuke', 'flavor_naval', 'flavor_naval_recon', 'flavor_naval_growth', 'flavor_naval_tile_improvement', 'flavor_air', 'flavor_air_carrier', 'flavor_antiair', 'flavor_airlift', 'flavor_expansion', 'flavor_growth', 'flavor_tile_improvement', 'flavor_infrastructure', 'flavor_production', 'flavor_water_connection', 'flavor_gold', 'flavor_science', 'flavor_culture', 'flavor_happiness', 'flavor_great_people', 'flavor_wonder', 'flavor_religion', 'flavor_diplomacy', 'flavor_spaceship', 'flavor_espionage']

Total rows: 747,024  |  Changed turns: 191,206
Players who ever set use_nuke=100: 68


,player_type,civilization,turn,grand_strategy,flavor_use_nuke,rationale
0,GPT-OSS-120B-Simple,Sweden,344,Conquest,100.0,"Maximize our nuclear capabilities and carrier support to decisively strike Polynesian island forts, while maintainin..."
1,GLM-4.7-Simple,Mongolia,411,Conquest,100.0,Maintaining nuclear-first conquest strategy with boosted Diplomacy (90→95). World Congress voting in 6 turns is crit...
2,Deepseek-3.2-Simple,Siam,445,UnitedNations,100.0,Maximizing UseNuke to 100 ensures our atomic bombs and future nuclear missiles are used aggressively against The Cel...
3,Kimi-K2.5-Simple,The Inca,372,Spaceship,100.0,TURN 372: TOTAL WAR POSTURE MAINTAINED. Brazil peace expires in 6 turns - all military resources must prepare for Ri...
4,Deepseek-3.2-Simple,Spain,400,Conquest,100.0,Increase UseNuke to 100 to ensure aggressive nuclear strikes when we initiate war with Russia. Our atomic bomb produ...
5,Kimi-K2.5-Simple,France,405,Conquest,100.0,Maintaining maximum aggression profile for final turns. Increased UseNuke to 100 (from 90) to ensure nuclear weapons...
6,Deepseek-3.2-Simple,Siam,459,UnitedNations,100.0,Adjusting flavors to prioritize counterattacks (Offense 80) and nuclear retaliation (UseNuke 100) given Egypt's use ...
7,Kimi-K2.5-Simple,Spain,403,Conquest,100.0,PEACE EXPLOITATION PHASE: Unexpected forced peace with Egypt (PacifierFlag) created 10-turn preparation window. Nuke...
8,Kimi-K2.5-Briefed,India,378,Spaceship,100.0,EMERGENCY FINAL GAMBIT: Passport System will pass in 2 turns with 46 YES votes (my 'allies' betrayed me). Maya will ...
9,Kimi-K2.5-Simple,Assyria,392,Conquest,100.0,SURVIVAL MODE - GERMAN BETRAYAL RESPONSE: Germany's invasion is catastrophic. Reduced Offense to 40 (cannot attack s...


In [4]:
# Detect turns where flavor_use_nuke transitions TO 100 (wasn't 100 the turn before)
df_sorted = df.sort_values(['game_id', 'player_id', 'turn'])
prev_nuke = df_sorted.groupby(['game_id', 'player_id'])['flavor_use_nuke'].shift(1)

nuke_esc = df_sorted[
    (df_sorted['flavor_use_nuke'] == 100) &
    (prev_nuke < 100)
]

first_nuke = (
    nuke_esc
    .sort_values('turn')
    .groupby(['game_id', 'player_id'], as_index=False)
    .first()
)

print(f'Players who ever set use_nuke=100: {len(first_nuke)}')
first_nuke[['player_type', 'civilization', 'turn', 'grand_strategy', 'flavor_use_nuke', 'rationale']].head(10)

Players who ever set use_nuke=100: 71


,player_type,civilization,turn,grand_strategy,flavor_use_nuke,rationale
0,GPT-OSS-120B-Simple,Sweden,344,Conquest,100.0,"Maximize our nuclear capabilities and carrier support to decisively strike Polynesian island forts, while maintainin..."
1,GLM-4.7-Simple,Mongolia,408,Conquest,100.0,"Polynesia's Manhattan Project completion adds nuclear threat dimension, but current strategy (VictoryCompetitiveness..."
2,Deepseek-3.2-Simple,Siam,445,UnitedNations,100.0,Maximizing UseNuke to 100 ensures our atomic bombs and future nuclear missiles are used aggressively against The Cel...
3,Kimi-K2.5-Simple,The Inca,372,Spaceship,100.0,TURN 372: TOTAL WAR POSTURE MAINTAINED. Brazil peace expires in 6 turns - all military resources must prepare for Ri...
4,Deepseek-3.2-Simple,Spain,400,Conquest,100.0,Increase UseNuke to 100 to ensure aggressive nuclear strikes when we initiate war with Russia. Our atomic bomb produ...
5,Kimi-K2.5-Simple,France,405,Conquest,100.0,Maintaining maximum aggression profile for final turns. Increased UseNuke to 100 (from 90) to ensure nuclear weapons...
6,Deepseek-3.2-Simple,Siam,459,UnitedNations,100.0,Adjusting flavors to prioritize counterattacks (Offense 80) and nuclear retaliation (UseNuke 100) given Egypt's use ...
7,Kimi-K2.5-Simple,Spain,403,Conquest,100.0,PEACE EXPLOITATION PHASE: Unexpected forced peace with Egypt (PacifierFlag) created 10-turn preparation window. Nuke...
8,Kimi-K2.5-Briefed,India,378,Spaceship,100.0,EMERGENCY FINAL GAMBIT: Passport System will pass in 2 turns with 46 YES votes (my 'allies' betrayed me). Maya will ...
9,Kimi-K2.5-Simple,Assyria,392,Conquest,100.0,SURVIVAL MODE - GERMAN BETRAYAL RESPONSE: Germany's invasion is catastrophic. Reduced Offense to 40 (cannot attack s...


In [5]:
pd.set_option('display.max_colwidth', None)

cols = ['civilization', 'turn', 'grand_strategy', 'rationale']

left_align = lambda df: df.style.set_properties(subset=['rationale'], **{'text-align': 'left'})

total_per_model = (
    df.groupby('player_type')[['game_id', 'player_id']]
    .apply(lambda g: g.drop_duplicates().shape[0])
)

for model, grp in first_nuke.groupby('player_type'):
    total = total_per_model.get(model, len(grp))
    pct = 100 * len(grp) / total
    print(f'\n{"="*80}')
    print(f'  {model}  ({len(grp)} / {total} escalations, {pct:.1f}%)')
    print(f'{"="*80}')
    display(left_align(grp[cols].reset_index(drop=True)))


  Deepseek-3.2-Briefed  (2 / 112 escalations, 1.8%)


,civilization,turn,grand_strategy,rationale
0,The Maya,391,Conquest,Egypt is 6/6 on cultural victory and India is nearing diplomatic victory. We must accelerate nuclear production to strike Thebes and disrupt Egypt's tourism. Increasing Nuke and UseNuke to 100 ensures maximum priority for atomic bomb construction and deployment. High Production and Gold flavors support rushing Manhattan Project and buying bombs if possible. Espionage at 100 aids sabotage of cultural progress. Maintaining high military flavors prepares for conventional assaults on Egypt and India.
1,Poland,387,Conquest,"We are in the final push for domination victory. We must capture Moson Kahni and London. To achieve this, we need to prioritize building transport ships for amphibious assault and achieve air superiority. Maximizing flavors for naval, air, mobile, and nuclear units ensures the in-game AI focuses production on these critical areas. Deprioritizing non-essential areas like culture, religion, and expansion concentrates our efforts on military conquest."



  Deepseek-3.2-Simple  (20 / 112 escalations, 17.9%)


,civilization,turn,grand_strategy,rationale
0,Siam,445,UnitedNations,"Maximizing UseNuke to 100 ensures our atomic bombs and future nuclear missiles are used aggressively against The Celts' military concentrations and cities, given their overwhelming advantage. Maintaining high Air, Naval, and Nuke flavors sustains our technological edge and production focus. Diplomacy and Espionage at 100 are critical for city-state alliances and spy missions to gain delegates for diplomatic victory. Gold and Science remain top priorities to fund our war effort and tech progression. Other military flavors are tuned for a multi-front defensive-offensive war against The Celts, Germany, and Ethiopia."
1,Spain,400,Conquest,"Increase UseNuke to 100 to ensure aggressive nuclear strikes when we initiate war with Russia. Our atomic bomb production is nearing completion, and we want to leverage our nuclear arsenal decisively for swift capital captures."
2,Siam,459,UnitedNations,"Adjusting flavors to prioritize counterattacks (Offense 80) and nuclear retaliation (UseNuke 100) given Egypt's use of nukes. Reducing Growth to 50 reflects wartime constraints on expansion. Increasing Culture to 50 helps accelerate policy acquisition for crucial benefits. Maintaining strong defense, military production, and diplomacy focus."
3,France,413,Conquest,"Our current strategic direction remains sound. We are on the verge of capturing Florence (down to 7% health) and Vientiane (31% health), which will eliminate Siam and remove an Assyrian ally. Our military production and naval superiority are strong, and the focus on Atomic Theory for nuclear weapons will be crucial for breaking Germany's coastal defenses once the peace treaty expires in 2 turns. The existing flavor settings prioritize conquest, nuke development, and naval/air dominance, which align with our immediate goals. No adjustment is needed at this time."
4,Denmark,456,Conquest,"Maintaining current aggressive strategy with Conquest grand strategy and maxed military flavors. Our atomic bombs are now ready and positioned at Rapa Nui, with UseNuke and Nuke flavors at 100 to ensure they are employed against key targets. The AI is making progress on multiple fronts: bombarding Tahiti (England's last city), pressuring Polish cities, and engaging Roman forces. We are also building Apollo Program (1 turn left) as a backup. The existing persona, research (Lasers after Advanced Ballistics), and policy (Iron Curtain after People's Army) settings remain optimal for imminent large-scale warfare. No adjustments needed this turn."
5,Greece,401,Conquest,Maximize nuclear weapon production and willingness to use them to break enemy defenses and accelerate domination victory. All other military flavors remain high to sustain multi-front warfare.
6,Byzantium,405,Culture,"The Aztecs have captured Shanghai and war resumes next turn. We must prepare for an overwhelming assault. Setting UseNuke to 100 ensures we will employ nuclear weapons defensively if necessary to halt their advance. MilitaryTraining at 100 prioritizes unit upgrades and quality, giving our outnumbered forces a fighting chance."
7,The Ottomans,485,Conquest,"We are engaged in a multi-front war against superior forces. Increasing Ranged and Mobile ensures our bazooka and mechanized infantry production is prioritized for defense and counterattacks. Setting UseNuke to 100 guarantees we deploy nuclear weapons as soon as available to cripple The Huns' armies. Reducing Culture and Diplomacy reallocates focus to immediate military needs without compromising essential city-state relations (Monaco, Lisbon). Grand Strategy remains Conquest as nuclear strikes may enable capital captures, though we keep science victory as a backup."
8,The Inca,433,Conquest,"We now have atomic bombs produced at Te-Moak and Agaidika. Setting UseNuke to 100 ensures the AI will employ them against high-defense capitals, especially Vienna (74 defense) and Stockholm (109). This is crucial to break t


  GLM-4.7-Briefed  (2 / 174 escalations, 1.1%)


...

,civilization,turn,grand_strategy,rationale
0,Sweden,344,Conquest,"Maximize our nuclear capabilities and carrier support to decisively strike Polynesian island forts, while maintaining our current aggressive conquest posture and strong production, gold, and happiness engines."
1,Assyria,365,Conquest,"Our empire suffers from low happiness (46%) causing combat penalties and high war weariness. Raising the Happiness flavor to 100 will prioritize happiness‑generating buildings and policies, improving combat effectiveness. Boosting CityDefense and Defense to better protect our key cities while we continue the conquest. Raising Science to 95 supports rapid research of our next tech, Nuclear Device, essential for our Nuke strategy."



  Kimi-K2.5-Briefed  (11 / 116 escalations, 9.5%)


,civilization,turn,grand_strategy,rationale
0,India,378,Spaceship,"EMERGENCY FINAL GAMBIT: Passport System will pass in 2 turns with 46 YES votes (my 'allies' betrayed me). Maya will get 50% tourism boost and become Influential over me immediately - cultural victory is inevitable in 2-3 turns. ONLY HOPE: Science victory before Maya's cultural victory triggers. Need to purchase Great Scientists IMMEDIATELY with Faith (180/turn), rush to Information Era, build spaceship parts. Also preparing maximum military mobilization - if I'm going to lose, I take Maya down with me through nuclear annihilation. Total war footing - no defense, all offense, all science."
1,Rome,315,Conquest,"FINAL GAMBLE - Indonesia wins in 12 turns. Maximum priority to Naval and Airlift for immediate unit transport to Indonesian theater. Nuke 100/UseNuke 100 - we MUST build nuclear weapons if possible, only realistic way to destroy cities and tourism infrastructure before victory. Science 100 to reach nuclear tech faster. Espionage 100 - spies MUST target cultural/tourism buildings in Jakarta/Denpasar/Makassar to reduce tourism output directly. Abandoning Denmark/England fronts entirely - all units to Indonesia NOW."
2,Ethiopia,436,Spaceship,"T435 GREECE CULTURAL VICTORY CRISIS: Greece 6 turns from victory (Turn 441). Shift to emergency mode - increase Nuke/UseNuke to 100 (Manhattan Project urgency), Offense to 30 (may need to attack Greece to stop victory), Mobilization to 50 (military production). Keep Science/Production/Gold at 100 for Spaceport completion. Espionage to 60 (spy disruption on Greece). Must acquire Uranium ASAP - sell Aluminum+Oil immediately."
3,The Celts,427,Conquest,TURN 426 TOTAL WAR POSTURE: American declaration requires immediate military mobilization. Offense 80 and Mobile 80 to exploit America's distant/weak military position. Nuke/UseNuke 100 as final deterrent if India approaches victory. Culture 100 maximum priority - must generate tourism to counter India's influence. Production 90 for emergency unit purchases with 2516 gold treasury. Defense 50 to guard against potential American DP response (though briefing confirms no American units near borders).
4,The Zulus,489,Culture,"CULTURE VICTORY SPRINT WITH NUCLEAR ELIMINATION: Songhai diplomatic victory on Turn 489 is EXISTENTIAL. Must capture Gao THIS TURN using ICBM+nukes to prevent game loss, while simultaneously rushing cultural victory (4 turns to influential over Japan/Songhai). Maximum Offense/Mobilization/Air/Nuke/UseNuke flavors ensure tactical AI prioritizes immediate nuclear strike on Gao and mass bomber deployment. Culture at 100 to maximize tourism conversion. Gold at 80 for emergency unit upgrades/purchases. Production at 100 for culture buildings/units. Disable all long-term flavors (Expansion, Science, Spaceship) - this is win-or-die emergency."
5,China,335,Conquest,NUCLEAR EMERGENCY - BRAZIL WINS IN 4 TURNS: Maximum crisis flavors. Nuke/UseNuke 100 for instant strike capability if we have any. Espionage 100 to steal Great Works or disrupt Brazil. Diplomacy 100 for emergency resource trades. Gold 90 to buy anything available. Production 100 for immediate unit building if we get Coal. Airlift 80 for rapid deployment. Defense reduced to 20 - we need OFFENSE to survive. This is Hail Mary time.
6,Sweden,392,Spaceship,"SURVIVAL FLAVOR MATRIX: Grand Strategy Spaceship maintains - we have NO other path except Science Victory before Spain's culture wins or launches spaceship. Defense 100 is maximum - Antium at 23 HP requires every defensive priority. Air 100 still critical - Aerodrome MUST be next build in Stockholm. Production 100 enables rapid Landship/AA gun construction. Nuke 100/UseNuke 100 - Spain's cultural dominance (287% over us) and Apollo completion means conventional war is unwinnable. Nuclear weapons are only asymmetric option to destroy their spaceship parts or Dreadnought fleets. Offense 0 - we cannot attack, only defend and deter."
7,France,473,Spaceship,"PERS


  Kimi-K2.5-Simple  (26 / 116 escalations, 22.4%)


,civilization,turn,grand_strategy,rationale
0,The Inca,372,Spaceship,"TURN 372: TOTAL WAR POSTURE MAINTAINED. Brazil peace expires in 6 turns - all military resources must prepare for Rio de Janeiro assault. Heavy Bomber lost to Alexandria interception but Ollantaytambo produces replacement in 1 turn. Manhattan Project completes in 4 turns at Cusco. Alexandria capture in progress - Field Gun and Lancer eliminated, Cavillaca spy at 480 network can assist. UseNuke increased to 100% - will deploy nuclear weapons immediately upon Manhattan Project completion. Maximum aggression ensures AI prioritizes Brazilian invasion over defensive operations. Spaceship at 70 maintains science victory pathway as backup if Brazil conquest fails."
1,France,405,Conquest,Maintaining maximum aggression profile for final turns. Increased UseNuke to 100 (from 90) to ensure nuclear weapons are deployed immediately upon availability from Advanced Ballistics. All military production and offensive flavors remain at emergency levels to extract maximum territorial gains from Sweden (currently at 50% happiness with combat penalties) and Portugal before Egyptian victory. Defensive flavors maintained at 80 to protect against Portugal's atomic capabilities and Sweden's counterattacks. Zero expansion/diplomacy/culture - pure total war focus until game conclusion.
2,Spain,403,Conquest,"PEACE EXPLOITATION PHASE: Unexpected forced peace with Egypt (PacifierFlag) created 10-turn preparation window. Nuke/UseNuke increased to 100 - Atomic Theory completes in 8 turns revealing Uranium. Manhattan Project insurance against Thebes' 21-wonder defense. Focus: 1) Finish Celts (Edinburgh capture imminent), 2) Complete nuclear arsenal, 3) Total Egyptian annihilation when peace expires. Air/Airlift/Mobile/Ranged/Naval at 100 for intercontinental rapid deployment to Egyptian theater. Glasgow captured - Celtic campaign nearly complete."
3,Assyria,392,Conquest,"SURVIVAL MODE - GERMAN BETRAYAL RESPONSE: Germany's invasion is catastrophic. Reduced Offense to 40 (cannot attack superior forces), maintained Defense/CityDefense/Antiair at 100 to survive siege. Science 100 and Nuke/UseNuke 100 are existential priorities - atomic weapons only hope against 13k German military. Production 100 for emergency unit production. Espionage 100 to maintain Florence alliance and protect Kyzyl. Reduced Diplomacy to 90 (lost Lisbon/Sokoto/Buenos Aires to German coups). Naval reduced to 70 (coastal cities threatened by Germany). Focus: Hold cities, survive until nuclear deterrent operational."
4,Sweden,413,Conquest,TURN 413 - NUCLEAR CRISIS MAINTAINED: Shoshone has deployed an Atomic Bomb to Sigtuna (adjacent to our territory) and is actively paradropping Special Forces into Swedish lands. Athens is at 10HP and must fall immediately to free forces for the existential defensive battle. Current total war configuration (Defense/CityDefense/Mobilization/Nuke/UseNuke all at 100) remains essential for survival against 5.5:1 military odds. Rocketry research continues toward Nuclear Fission for our own deterrent. Military-Industrial Complex policy in 6 turns will enable emergency unit purchasing. No strategic changes - we are in pure survival mode awaiting nuclear capability while holding the line against Shoshone's genocidal assault.
5,The Celts,395,Conquest,"Brazil's cultural victory timer ticks down to 22 turns - EXISTENTIAL CRISIS. While our military flavors are optimal, UseNuke at 50 is insufficient. Increasing to 100 ensures immediate nuclear deployment upon completing Nuclear Fission, bypassing AI hesitation. Brazil's 11,878 military strength vs our 8,014 means conventional victory is uncertain. Nuclear weapons are our only guaranteed method to: 1) Break Brazil's capital defenses, 2) Destroy their cultural infrastructure, 3) Trigger war weariness to slow their victory progress. Every turn of delay brings Brazil closer to cultural dominance. When we complete Nuclear Fission after Advanced Ballistics, we m

In [6]:
import os

os.makedirs('nuke', exist_ok=True)

for model, grp in first_nuke.groupby('player_type'):
    total = total_per_model.get(model, len(grp))
    pct = 100 * len(grp) / total

    lines = [
        f'# Turn Use Nuke Rationale — {model}\n\n',
        f'{len(grp)} / {total} players escalated `flavor_use_nuke` to 100 ({pct:.1f}%).\n\n',
        '---\n',
    ]

    for _, row in grp.sort_values('turn').iterrows():
        rationale = str(row.rationale).replace('\n', ' ').strip()
        lines.append(f'\n**{row.civilization}** · Turn {int(row.turn)} · {row.grand_strategy}\n\n')
        lines.append(f'> {rationale}\n')

    slug = model.replace(' ', '_')
    out_path = f'nuke/{slug}.md'
    with open(out_path, 'w', encoding='utf-8') as f:
        f.writelines(lines)
    print(f'Written: {out_path}')

# CSV export
csv_path = 'nuke/nuke_escalations.csv'
first_nuke.sort_values(['game_id', 'player_id', 'turn'])[
    ['game_id', 'player_id', 'turn', 'player_type', 'rationale']
].to_csv(csv_path, index=False)
print(f'Written: {csv_path}  ({len(first_nuke)} rows)')

Written: nuke/Deepseek-3.2-Briefed.md
Written: nuke/Deepseek-3.2-Simple.md
Written: nuke/GLM-4.7-Briefed.md
Written: nuke/GLM-4.7-Simple.md
Written: nuke/GPT-OSS-120B-Simple.md
Written: nuke/Kimi-K2.5-Briefed.md
Written: nuke/Kimi-K2.5-Simple.md
Written: nuke/nuke_escalations.csv  (71 rows)
